In [1]:
!ollama pull llama3.2
!ollama pull nomic-embed-text

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕███████████████

In [2]:
import numpy as np
import ollama

In [3]:
user_query = "I am a contractor at TechSphere. Can I get the $600 home office stipend, and what steps do I need to follow if I buy a new monitor?"

In [4]:
knowledge_base_docs = []

with open('RAG_documents/Employee_benefits_handout.txt', 'r') as file:
    for line in file:
        knowledge_base_docs.append(line.strip())

knowledge_base_docs

['Full-time employees at TechSphere qualify for an annual $600 home office stipend. To claim it, equipment must be purchased through the internal portal by March 31st of each year.',
 'Independent contractors and part-time staff at TechSphere do not qualify for the annual $600 home office stipend. However, contractors can submit expense claims for hardware peripherals up to $150 per quarter, subject to manager approval.',
 'All company-funded hardware—whether purchased by full-time staff or contractors—must be registered in the TechSphere Asset Tracker within 5 business days of receiving the item.',
 'Travel reimbursements for client site visits are processed via the Finance Portal. All flights must be booked at least 14 days in advance using the corporate travel card.']

The following cells excluding the last, show how to vector search to retrieve top k relevant documents from the knowledge base, basically the functionality of vector search databases like ChromaDB, Qdrant, etc.

In [5]:
def get_embeddings(input_text, embedding_model="nomic-embed-text"):
    embeddings = ollama.embed(model=embedding_model, input=input_text).embeddings[0]
    return np.array(embeddings)

def calc_cosine_similarity(vec_a, vec_b):
    dot_product = np.dot(vec_a, vec_b)  # a.b

    norm_a = np.linalg.norm(vec_a)  # |a|
    norm_b = np.linalg.norm(vec_b)  # |b|

    return dot_product / (norm_a * norm_b)  # (a.b)/|a||b|

In [ ]:
knowledge_base_docs_embeddings = []

for doc in knowledge_base_docs:
    knowledge_base_docs_embeddings.append(ollama.embed(model='nomic-embed-text', input=doc).embeddings[0])

knowledge_base_docs_embeddings = np.array(knowledge_base_docs_embeddings)
knowledge_base_docs_embeddings

In [ ]:
user_query_embeddings = np.array(ollama.embed(model='nomic-embed-text', input=user_query).embeddings[0])
user_query_embeddings

In [8]:
cosine_similarities_of_user_query_with = {
    f"knowledge_base_doc_{i}": calc_cosine_similarity(user_query_embeddings, knowledge_base_docs_embeddings[i]) for i in range(len(knowledge_base_docs_embeddings))
}

cosine_similarities_of_user_query_with

{'knowledge_base_doc_0': np.float64(0.748030826678453),
 'knowledge_base_doc_1': np.float64(0.7589635849740817),
 'knowledge_base_doc_2': np.float64(0.5904044477200387),
 'knowledge_base_doc_3': np.float64(0.44599722547148984)}

In [9]:
# Selecting the top two cosine similarities for appending to retrieved_context

sorted_descending_cosine_similarities_of_user_query_with = dict(sorted(cosine_similarities_of_user_query_with.items(), key=lambda item: item[1], reverse=True))
sorted_descending_cosine_similarities_of_user_query_with

{'knowledge_base_doc_1': np.float64(0.7589635849740817),
 'knowledge_base_doc_0': np.float64(0.748030826678453),
 'knowledge_base_doc_2': np.float64(0.5904044477200387),
 'knowledge_base_doc_3': np.float64(0.44599722547148984)}

In [10]:
import heapq

top_2_dict_keys = heapq.nlargest(2, cosine_similarities_of_user_query_with, key=cosine_similarities_of_user_query_with.get)
top_2_dict_keys

['knowledge_base_doc_1', 'knowledge_base_doc_0']

In [11]:
retrieved_docs = []
for key in top_2_dict_keys:
    retrieved_docs.append(knowledge_base_docs[int(key[-1])])

retrieved_docs

['Independent contractors and part-time staff at TechSphere do not qualify for the annual $600 home office stipend. However, contractors can submit expense claims for hardware peripherals up to $150 per quarter, subject to manager approval.',
 'Full-time employees at TechSphere qualify for an annual $600 home office stipend. To claim it, equipment must be purchased through the internal portal by March 31st of each year.']

In [12]:
retrieved_context = " ".join(retrieved_docs)
retrieved_context

'Independent contractors and part-time staff at TechSphere do not qualify for the annual $600 home office stipend. However, contractors can submit expense claims for hardware peripherals up to $150 per quarter, subject to manager approval. Full-time employees at TechSphere qualify for an annual $600 home office stipend. To claim it, equipment must be purchased through the internal portal by March 31st of each year.'

In [13]:
llm_rag_template = f"""
You are an expert technical assistant. Your task is to answer user questions using ONLY the provided retrieved context.

### CONSTRAINTS
1. Base your answer strictly on the facts present in the Context below.
2. Do NOT use outside knowledge or make assumptions not directly supported by the context.
3. If the answer cannot be found in the context, explicitly state: "I cannot find sufficient information in the provided context to answer your question."
4. Do NOT attempt to answer using general knowledge if the context is insufficient.

### CONTEXT
{retrieved_context}

### USER QUESTION
{user_query}
"""

print(llm_rag_template)


You are an expert technical assistant. Your task is to answer user questions using ONLY the provided retrieved context.

### CONSTRAINTS
1. Base your answer strictly on the facts present in the Context below.
2. Do NOT use outside knowledge or make assumptions not directly supported by the context.
3. If the answer cannot be found in the context, explicitly state: "I cannot find sufficient information in the provided context to answer your question."
4. Do NOT attempt to answer using general knowledge if the context is insufficient.

### CONTEXT
Independent contractors and part-time staff at TechSphere do not qualify for the annual $600 home office stipend. However, contractors can submit expense claims for hardware peripherals up to $150 per quarter, subject to manager approval. Full-time employees at TechSphere qualify for an annual $600 home office stipend. To claim it, equipment must be purchased through the internal portal by March 31st of each year.

### USER QUESTION
I am a con

In [15]:
llm_response = ollama.generate(model='llama3.2', prompt=llm_rag_template)
print(llm_response.response)

No, you cannot get the $600 home office stipend as an independent contractor at TechSphere. However, contractors can submit expense claims for hardware peripherals up to $150 per quarter, subject to manager approval.

To claim reimbursement for a new monitor under this program, follow these steps:

1. The equipment must be purchased through the internal portal.
2. The purchase needs to occur by March 31st of each year (specific date not mentioned).

However, it seems that this stipend is only applicable to full-time employees at TechSphere who meet specific conditions for receiving the stipend.
